In [11]:
import re
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
import stanza

In [8]:
def extract_criteria_blocks(text):
    # Rozdelenie podľa štruktúr typu "Položka č. X" alebo "kritérium č. X"
    blocks = re.split(r"(?i)(?:položka|kritérium)\s*č\.\s*\d+[:\-]?\s*", text)
    # Vyčistenie a odstránenie prázdnych/krátkych fragmentov
    return [b.strip() for b in blocks if len(b.strip()) > 10]

def normalize_criterion_text(text):
    # Odstráni opakujúce sa štruktúry ako "Kritérium č. 7 -"
    text = re.sub(r"(?i)kritérium\s*č\.\s*\d+[:\-]?\s*", "", text)
    text = re.sub(r"(?i)položka\s*č\.\s*\d+[:\-]?\s*", "", text)
    # Odstránenie čísel vlastností (napr. "vlastnosť č. 1.16")
    text = re.sub(r"(?i)vlastnosť\s*č\.\s*[\d.]+", "", text)
    return text.strip()

In [15]:
# 1. Načítanie pôvodného datasetu
df_raw = pd.read_csv("contract_criteria_export.csv")
all_rows = []

# 2. Rozdelenie každého riadku podľa kritérií
for idx, row in df_raw.iterrows():
    entry = str(row["criterion"])
    blocks = extract_criteria_blocks(entry)
    
    for block in blocks:
        norm_block = normalize_criterion_text(block)
        if norm_block and len(norm_block.split()) > 3:
            new_row = row.copy()
            new_row["criterion"] = norm_block
            all_rows.append(new_row)

# 3. Vytvorenie nového DataFrame so všetkými pôvodnými atribútmi
df_clean = pd.DataFrame(all_rows)

# 4. Odstrániť úplné duplicity (rovnaké vo všetkých stĺpcoch)
# df_clean = df_clean.drop_duplicates()

# 5. Uloženie
df_clean.to_csv("criteria_data_cleaned.csv", index=False)

In [ ]:
# model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
# all_criteria = df_clean["criterion"].tolist()

# all_embeddings = model.encode(all_criteria, convert_to_tensor=False)

# np.save("criteria_embeddings2.npy", all_embeddings)
# df_clean.to_csv("criteria_data_final.csv", index=False)

c:\Users\marek\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [16]:
# Inicializácia Stanza (ak to potrebuješ na ďalšie spracovanie)
stanza.download('sk')
nlp = stanza.Pipeline(lang='sk', processors='tokenize')

# Načítanie modelu a dát
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
df2 = pd.read_csv("criteria_data_final.csv")  # nový súbor so spracovanými dátami

# Výber textových kritérií
all_criteria = df["criterion"].fillna("").astype(str).tolist()

# Výpočet embeddingov
all_embeddings = model.encode(all_criteria, convert_to_tensor=False)

# Uloženie embeddingov a dát
np.save("criteria_embeddings2.npy", all_embeddings)
df2.to_csv("criteria_data_final.csv", index=False)

2025-05-29 13:26:12 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-05-29 13:26:12 INFO: Downloading default packages for language: sk (Slovak) ...
2025-05-29 13:26:13 INFO: File exists: C:\Users\marek\stanza_resources\sk\default.zip
2025-05-29 13:26:14 INFO: Finished downloading models and saved to C:\Users\marek\stanza_resources
2025-05-29 13:26:14 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-05-29 13:26:14 INFO: Downloaded file to C:\Users\marek\stanza_resources\resources.json
2025-05-29 13:26:14 WARNING: Language sk package default expects mwt, which has been added
2025-05-29 13:26:14 INFO: Loading these models for language: sk (Slovak):
| Processor | Package |
-----------------------
| tokenize  | snk     |
| mwt       | snk     |

2025-05-29 13:26:14 INFO: Using device: cpu
2025-05-29 13:26:14 INFO: 

In [14]:
df2.contract_id.nunique()

1396